In [ ]:
#Def path to folder
path = "Files/bronze/online-retail-dataset.csv"

#Read file with spark 
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path)

#Print top 10
display(df.limit(10))


StatementMeta(, 512c3f6f-0659-4308-ab9a-9d8f10716c91, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, d16dda80-5334-4b6e-82a8-9475c8fbd57b)

In [ ]:
from pyspark.sql.functions import col, to_date

# 1. ESTA ES LA SOLUCIÓN AL ERROR: 
# Le decimos a Spark que sea flexible con el formato de fecha antiguo
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# 2. Limpieza inicial: Quitamos filas sin ID de cliente
df_clean = df.filter(col("CustomerID").isNotNull())

# 3. Transformación: 
# Usamos 'M/d/yyyy H:mm' porque el error mostró que viene un solo dígito (6/12...)
df_silver = df_clean.withColumn("InvoiceDate", to_date(col("InvoiceDate"), "M/d/yyyy H:mm"))

# 4. Guardar en la Capa Plata
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_retail_dataset")

print("Now it is meant to succeed, hopefully")

StatementMeta(, 512c3f6f-0659-4308-ab9a-9d8f10716c91, 5, Finished, Available, Finished)

Now it is meant to succeed, hopefully


In [ ]:
# 1. Creamos una vista temporal para poder usar SQL puro
df_silver.createOrReplaceTempView("vista_silver")

# 2. Usamos SQL para agrupar y sumar
df_gold = spark.sql("""
    SELECT 
        CustomerID, 
        ROUND(SUM(Quantity * UnitPrice), 2) as TotalSpent,
        COUNT(InvoiceNo) as TotalTransactions
    FROM vista_silver
    GROUP BY CustomerID
    ORDER BY TotalSpent DESC
""")

# 3. Guardamos la tabla final de negocio
df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_spend_per_client")

display(df_gold.limit(10))

StatementMeta(, 512c3f6f-0659-4308-ab9a-9d8f10716c91, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3b82f8a7-9788-4c98-8b2f-54c5976a7465)